# Очистка данных: Deals

In [1]:
import os
import re
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import help_130625_dam as h

DATA_PATH = os.path.join('..', 'Sources', 'Deals (Done).xlsx')
OUT_PATH  = os.path.join('..', 'data', 'cleaned', 'deals_clean.pkl')

## Загрузка и первичный осмотр

In [2]:
# Contact Name читаем как str, чтобы избежать
# потери точности при промежуточном float64 (проявляется при наличии NaN в колонке)
df = pd.read_excel(DATA_PATH, dtype={'Contact Name': str})

# Переименование столбцов в snake_case
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

# Переименование contact_name в contact_id для единообразия с другими таблицами
df = df.rename(columns={'contact_name': 'contact_id'})

n_before = len(df)
print(f'Форма: {df.shape}')

df.head()

Форма: (21595, 23)


,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,product,education_type,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch
0,5.805028e+18,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,5805028000056849495,NaN,NaN
1,5.805028e+18,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Web Developer,Morning,21.06.2024 15:23,6.0,NaN,0,2000,5805028000056834471,NaN,NaN
2,5.805028e+18,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,00:26:43,b1-at,...,NaN,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,5805028000056854421,NaN,NaN
3,5.805028e+18,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,01:00:04,bloggersvideo14com,...,NaN,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,5805028000056889351,NaN,NaN
4,5.805028e+18,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,00:53:12,website,...,NaN,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,5805028000056876176,NaN,NaN


In [96]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21595 entries, 0 to 21594
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   21593 non-null  float64
 1   deal_owner_name      21564 non-null  str    
 2   closing_date         14645 non-null  str    
 3   quality              19340 non-null  str    
 4   stage                21593 non-null  str    
 5   lost_reason          16124 non-null  str    
 6   page                 21593 non-null  str    
 7   campaign             16067 non-null  str    
 8   sla                  15533 non-null  object 
 9   content              14147 non-null  str    
 10  term                 12454 non-null  str    
 11  source               21593 non-null  str    
 12  payment_type         496 non-null    str    
 13  product              3592 non-null   str    
 14  education_type       3300 non-null   str    
 15  created_time         21593 non-null  str    
 1

In [97]:
h.descr_df(df, include='all', show_sample_rows=True)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,float64,21593,2,8612,5805028000056864768.0,5805028000056859648.0,5805028000056832000.0,5.805028e+18,5.805028e+18,5.805028e+18,5.805028e+18
1,deal_owner_name,str,21564,31,27,Ben Hall,Ulysses Adams,Ulysses Adams,NaN,NaN,NaN,NaN
2,closing_date,str,14645,6950,359,NaN,NaN,21.06.2024,NaN,NaN,NaN,NaN
3,quality,str,19340,2255,6,NaN,NaN,D - Non Target,NaN,NaN,NaN,NaN
4,stage,str,21593,2,13,New Lead,New Lead,Lost,NaN,NaN,NaN,NaN
5,lost_reason,str,16124,5471,21,NaN,NaN,Non target,NaN,NaN,NaN,NaN
6,page,str,21593,2,34,/eng/test,/at-eng,/at-eng,NaN,NaN,NaN,NaN
7,campaign,str,16067,5528,154,03.07.23women,NaN,engwien_AT,NaN,NaN,NaN,NaN
8,sla,object,15533,6062,13357,NaN,NaN,00:26:43,NaN,NaN,NaN,NaN
9,content,str,14147,7448,187,v16,NaN,b1-at,NaN,NaN,NaN,NaN


In [98]:
full_dupes = df.duplicated().sum()
print(f'Полных дубликатов: {full_dupes}')

Полных дубликатов: 2


## 2. Дедупликация

> **Примечание**: одна и та же сделка (одинаковый `Id`) может появляться несколько раз — 
> каждая запись отражает отдельное событие в воронке (смену Stage). Это не дубликат, 
> а **история продвижения сделки**. Удаляем только полностью идентичные строки.

In [3]:
# Удаляем технический шум (полностью пустые строки без Id)
df = df.dropna(subset=['id']).reset_index(drop=True)
print(f'Строк после удаления пустых Id: {len(df)}')

Строк после удаления пустых Id: 21593


In [4]:
full_dupes = df.duplicated().sum()
print(f'Полных дубликатов строк: {full_dupes}')

id_dupes = df.duplicated(subset='id').sum()
print(f'Строк с повторяющимся Id: {id_dupes}  (история воронки — сохраняем)')

if full_dupes > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Строк после дедупликации: {len(df)}')

Полных дубликатов строк: 2
Строк с повторяющимся Id: 12981  (история воронки — сохраняем)
Строк после дедупликации: 21591


In [101]:
# Пропущенные значения
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
display(
    pd.DataFrame({'Пропуски': missing, '% пропусков': missing_pct})
    .query('Пропуски > 0')
    .sort_values('Пропуски', ascending=False)
)
print('Строк без пропусков:', df.dropna().shape[0])

,Пропуски,% пропусков
payment_type,21095,97.70
months_of_study,20751,96.11
level_of_deutsch,20340,94.21
city,19080,88.37
education_type,18292,84.72
course_duration,18004,83.39
product,17999,83.36
initial_amount_paid,17426,80.71
offer_total_amount,17406,80.62
term,9137,42.32


Строк без пропусков: 3


## 3. Типы данных: даты

In [5]:
# Created Time: '21.06.2024 15:30'  → datetime
df['created_time'] = pd.to_datetime(df['created_time'], dayfirst=True, errors='coerce')

# Closing Date: '21.06.2024' → datetime (NaT = сделка ещё открыта)
df['closing_date'] = pd.to_datetime(df['closing_date'], dayfirst=True, errors='coerce')

print('created_time:', df['created_time'].dtype, '| NaT:', df['created_time'].isna().sum())
print('closing_date:', df['closing_date'].dtype, '| NaT:', df['closing_date'].isna().sum())
print(f'Диапазон created_time: {df["created_time"].min()}  →  {df["created_time"].max()}')

created_time: datetime64[us] | NaT: 0
closing_date: datetime64[us] | NaT: 6947
Диапазон created_time: 2023-07-03 17:03:00  →  2024-06-21 15:30:00


## Валидация: Дата регистрации vs Дата создания сделки

> Проверка бизнес-логики: клиент не может создать сделку раньше, чем он зарегистрировался в системе. 
> Для этого подтянем `Created Time` из таблицы контактов.

In [6]:
CONTACTS_CLEAN = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')

if os.path.exists(CONTACTS_CLEAN):
    contacts = pd.read_pickle(CONTACTS_CLEAN)
    
    # Чтобы не множить строки при merge, гарантируем уникальность ID в контактах
    contacts_lookup = (
        contacts[['id', 'created_time']]
        .drop_duplicates(subset='id')
        .rename(columns={'created_time': 'lead_created_time'})
    )
    # Для validation merge приводим оба ключа к str (contact_id пока str, id — int64)
    contacts_lookup['id_str'] = contacts_lookup['id'].astype(str)
    
    tmp_m = df.merge(
        contacts_lookup,
        left_on='contact_id',   # Теперь используем новое имя колонки
        right_on='id_str',
        how='left'
    )
    
    # Поиск аномалий: регистрация лида ПОЗЖЕ даты создания сделки
    anomalies_mask = tmp_m['lead_created_time'] > tmp_m['created_time']
    n_anomalies = anomalies_mask.sum()
    matched = tmp_m['lead_created_time'].notna().sum()
    print(f'Сделок с привязанным контактом: {matched} из {len(tmp_m)}')
    print(f'Найдено аномалий (регистрация позже сделки): {n_anomalies}')
    
    if n_anomalies > 0:
        display(tmp_m[anomalies_mask][['contact_id', 'created_time', 'lead_created_time']].head())
        print("Аномалии зафиксированы.")
else:
    print('Файл contacts_clean.pkl не найден. Сначала выполните блокнот 01_cleaning_contacts.')

Сделок с привязанным контактом: 21529 из 21591
Найдено аномалий (регистрация позже сделки): 23


,contact_id,created_time,lead_created_time
3725,5805028000050513155,2024-04-29 22:39:00,2024-05-17 13:02:00
11119,5805028000029660580,2024-02-01 18:22:00,2024-02-01 18:28:00
11131,5805028000029661053,2024-02-01 16:08:00,2024-02-01 16:46:00
11372,5805028000028751715,2024-01-29 13:49:00,2024-01-29 17:59:00
11721,5805028000038864187,2024-01-25 10:14:00,2024-03-20 17:44:00


Аномалии зафиксированы.


## 4. SLA: время ответа → секунды

> `SLA` хранит объекты `datetime.time` (hh:mm:ss). Для анализа удобнее хранить как **целое число секунд**.

In [7]:
print('sla — тип и примеры значений до обработки:')
print(df['sla'].dtype)

def time_to_seconds(t):
    """Конвертирует datetime.time → float (секунды) или np.nan."""
    if isinstance(t, datetime.time):
        return float(t.hour * 3600 + t.minute * 60 + t.second)
    try:
        # Пытаемся сконвертировать, если это уже число или строка
        val = float(t)
        return val if np.isfinite(val) else np.nan
    except (ValueError, TypeError):
        return np.nan

# 1. Принудительная конвертация всего столбца в float
df['sla'] = df['sla'].apply(time_to_seconds)

# 2. Фильтрация экстремальных выбросов (> 48 часов)
MAX_SLA = 48 * 3600 
outliers_mask = df['sla'] > MAX_SLA
if outliers_mask.any():
    print(f"Обнаружено {outliers_mask.sum()} аномально высоких значений SLA (>48ч), сбрасываем их.")
    df.loc[outliers_mask, 'sla'] = np.nan

# 3. Восполнение медианой по менеджеру
if 'deal_owner_name' in df.columns:
    # Используем transform для получения вектора медиан того же размера, что и df
    manager_medians = df.groupby('deal_owner_name', observed=True)['sla'].transform('median')
    global_median = df['sla'].median()
    
    n_nan_before = df['sla'].isna().sum()
    # Последовательно заполняем пропуски
    df['sla'] = df['sla'].fillna(manager_medians).fillna(global_median)
    n_filled = n_nan_before - df['sla'].isna().sum()
    print(f'Восполнено пропусков SLA: {n_filled}')

# 4. Финальное приведение к целочисленному типу с поддержкой NULL
df['sla'] = df['sla'].round(0).astype('Int32')

print(f'\nsla после обработки — тип: {df["sla"].dtype}')
print(f'Осталось NaN в SLA: {df["sla"].isna().sum()}')
if df["sla"].notna().any():
    print(f'Диапазон: {df["sla"].min()} сек  →  {df["sla"].max()} сек')
    print(f'Среднее: {df["sla"].mean():.0f} сек ({df["sla"].mean()/60:.1f} мин)')
    print(f'Медиана: {df["sla"].median():.0f} сек ({df["sla"].median()/60:.1f} мин)')
else:
    print("Данные SLA отсутствуют.")

sla — тип и примеры значений до обработки:
object
Восполнено пропусков SLA: 7919

sla после обработки — тип: Int32
Осталось NaN в SLA: 0
Диапазон: 3 сек  →  86319 сек
Среднее: 21904 сек (365.1 мин)
Медиана: 14344 сек (239.1 мин)


## 5. Числовые поля: очистка сумм

In [8]:
def clean_amount(series):
    """Убирает знак €, пробелы, меняет европейский разделитель (3.500,00 → 3500.00)."""
    if series.dtype == object:
        series = (
            series.astype(str)
            .str.replace(r'[€$£\s\xa0]', '', regex=True)   # убираем валюту и пробелы
            .str.replace(r'\.(\d{3})', r'\1', regex=True)   # '3.500' → '3500' (тыс. разделитель)
            .str.replace(',', '.', regex=False)              # '3500,00' → '3500.00'
        )
        return pd.to_numeric(series, errors='coerce').fillna(0) # Заполняем пропуски нулями
    return pd.to_numeric(series, errors='coerce').fillna(0)

AMOUNT_COLS = ['initial_amount_paid', 'offer_total_amount']
for col in AMOUNT_COLS:
    df[col] = clean_amount(df[col])
    print(f'{col}: {df[col].dtype}, min={df[col].min()}, max={df[col].max():,.0f}, NaN={df[col].isna().sum()}')


initial_amount_paid: float64, min=0.0, max=11,500, NaN=0
offer_total_amount: float64, min=0.0, max=11,500, NaN=0


## 6. Id и Contact Id: object → Int64 (безопасно)

In [9]:
# id: используем pd.to_numeric — id хранится как число в Excel, openpyxl читает его как
# Python int (без потери точности), pd.to_numeric его не трогает.
df['id'] = pd.to_numeric(df['id'], errors='coerce').astype('Int64')

# contact_id: ВАЖНО — читаем как str, но pd.to_numeric идёт через float64 и ОКРУГЛЯЕТ
# 19-значные числа (5805028000056849495 → 5805028000056849408).
# Используем Python int() напрямую (str → int без float64-промежутка).
def _str_to_int64(v):
    """Точное преобразование строки/числа в Python int без потери precision через float64."""
    if pd.isna(v):
        return pd.NA
    s = str(v).split('.')[0].strip()  # убираем '.0' у float-строк
    if s in ('', 'nan', 'None'):
        return pd.NA
    return int(s)

df['contact_id'] = pd.array([_str_to_int64(v) for v in df['contact_id']], dtype='Int64')

print('id:        ', df['id'].dtype, '| NaN:', df['id'].isna().sum())
print('contact_id:', df['contact_id'].dtype, '| NaN:', df['contact_id'].isna().sum())
if not df['id'].dropna().empty:
    print('Пример id:', df['id'].dropna().iloc[0])
if not df['contact_id'].dropna().empty:
    cid_ex = df['contact_id'][df['contact_id'] > 0].dropna().iloc[0]
    print('Пример contact_id:', cid_ex, '| кратно 2048:', int(cid_ex) % 2048 == 0)

id:         Int64 | NaN: 0
contact_id: Int64 | NaN: 61
Пример id: 5805028000056864768
Пример contact_id: 5805028000056849495 | кратно 2048: False


## 7. Level of Deutsch: нормализация

> Поле содержит 215 уникальных значений: смесь кириллицы и латиницы (например `а2` vs `A2`, `б1` vs `B1`),
> а также свободный текст (адреса, фразы). Приводим к стандарту CEFR (A0–C2), остальное → `Unknown`.

In [10]:
# Список всех значений, которые не соответствуют паттерну [A1-C2] после очистки кириллицы
CYR_TO_LAT = str.maketrans('абвсАБВС', 'abvcABVC')

# ТАБЛИЦА ЗАМЕН: 
LEVEL_MAP = {
    'а': 'a', 'А': 'A',
    'б': 'b', 'Б': 'B',
    'в': 'b', 'В': 'B',
    'с': 'c', 'С': 'C'
}

def normalize_deutsch(value):
    if pd.isna(value):
        return pd.NA
    
    orig_s = str(value).strip()
    s_lower = orig_s.lower()
    
    # 1. Специальные случаи для A0
    a0_exact = ['0', 'no', 'none', '?', '-', 'нет', 'a'] # A без цифры -> A0
    a0_keywords = ['никакой', 'нулевой', 'не учил', 'не учила', 'anfanger', 'beginner', 'начальный']
    if s_lower in a0_exact or any(keyword in s_lower for keyword in a0_keywords):
        return 'A0'
        
    # 2. Специальные случаи для других уровней
    if s_lower == 'в': return 'B1'
    if s_lower == 'f2': return 'A2'
    if s_lower == 'c': return 'C1'
    
    # 3. ПОИСК УРОВНЕЙ: Любая буква [AaBbCcАаБбВвСс] + цифра [012]
    match = re.search(r'([AaBbCcАаБбВвСс][0-2])', orig_s)
    if match:
        found = match.group(1)
        letter = found[0]
        digit = found[1]
        letter_lat = LEVEL_MAP.get(letter, letter).upper()
        return f"{letter_lat}{digit}"
    
    # 4. Маппинг остальных ключевых слов
    words_map = {
        'intermediate': 'B1', 'средний': 'B1',
        'advanced': 'C1'
    }
    for word, level in words_map.items():
        if word in s_lower:
            return level
            
    return 'Unclear'

# Применяем очистку
# Используем df_raw, чтобы не накапливать ошибки при повторных запусках
df_raw = pd.read_excel(DATA_PATH)
df['level_of_deutsch'] = df_raw['Level of Deutsch'].apply(normalize_deutsch)

# Анализ оставшихся нестандартных значений (для проверки)
def get_non_standard(value):
    if pd.isna(value) or value == 'Unclear': return value
    # Если значение уже в стандарте A1-C2, возвращаем None
    if re.search(r'\b([AaBbCc][012])\b', str(value)):
        return None
    return value

non_std_count = (df['level_of_deutsch'] == 'Unclear').sum()
print(f"Всего строк с нераспознанным уровнем (Unclear): {non_std_count}")

print("\n--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---")
display(df['level_of_deutsch'].value_counts().to_frame())

Всего строк с нераспознанным уровнем (Unclear): 16

--- ИТОГОВОЕ РАСПРЕДЕЛЕНИЕ УРОВНЕЙ ---


,count
level_of_deutsch,
B1,818
B2,171
A2,150
A0,40
C1,28
A1,25
Unclear,16
C2,3


## 7. Восполнение пропусков (Backfill) на основе Contact Name

> Если для одного и того же клиента (`Contact Name`) в разных сделках заполнены разные поля (Source, City и т.д.), 
> мы можем «протянуть» эти значения на пустые строки этого же клиента.

In [11]:
# Анализ анонимных сделок (без привязанного контакта) в разрезе менеджеров
# Выполняем ДО замены NaN на -1, чтобы видеть реальную картину пропусков
no_contact_mask = df['contact_id'].isna()

if no_contact_mask.any():
    print(f"Всего сделок без контакта: {no_contact_mask.sum()}")
    print(f"Общая сумма оплат в них: {df.loc[no_contact_mask, 'initial_amount_paid'].sum():,.2f} €")
    
    print("\n--- РАСПРЕДЕЛЕНИЕ АНОНИМНЫХ СДЕЛОК ПО МЕНЕДЖЕРАМ ---")
    owner_analysis = (
        df[no_contact_mask]
        .groupby('deal_owner_name', observed=False)
        .agg(
            Кол_во_сделок=('id', 'count'),
            Сумма_оплат=('initial_amount_paid', 'sum')
        )
    )
    owner_analysis = owner_analysis[owner_analysis['Кол_во_сделок'] > 0].sort_values('Сумма_оплат', ascending=False)
    display(owner_analysis)
else:
    print("Сделок без сопоставленного контакта не найдено.")


Всего сделок без контакта: 61
Общая сумма оплат в них: 20,200.00 €

--- РАСПРЕДЕЛЕНИЕ АНОНИМНЫХ СДЕЛОК ПО МЕНЕДЖЕРАМ ---


,Кол_во_сделок,Сумма_оплат
deal_owner_name,,
Victor Barnes,7,5800.0
Charlie Davis,11,5200.0
Kevin Parker,11,4200.0
Ian Miller,3,3000.0
Quincy Vincent,7,2000.0
Diana Evans,2,0.0
Julia Nelson,2,0.0
John Doe,3,0.0
Jane Smith,3,0.0


In [12]:
# Поля для заполнения
COLS_TO_FILL = ['source', 'campaign', 'city', 'level_of_deutsch', 'deal_owner_name']
COLS_CHECK = COLS_TO_FILL + (['course_duration'] if 'course_duration' in df.columns else [])

# 1. Подготовка: Группировка сделок БЕЗ contact_id под одним ID -1
# Это сохраняет деньги в системе, даже если контакт не привязан.
no_contact_mask = df['contact_id'].isna()
df.loc[no_contact_mask, 'contact_id'] = -1

# Приводим колонки к object перед восполнением, чтобы избежать конфликтов Categorical
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df[col].astype(object)

# Сохраняем состояние до восполнения
missing_before = df[COLS_CHECK].isnull().sum()

# 2. Backfill по contact_id
df = df.sort_values(['contact_id', 'created_time'])
for col in COLS_TO_FILL:
    if col in df.columns:
        df[col] = df.groupby('contact_id', group_keys=False)[col].apply(lambda x: x.ffill().bfill())

# 3. Восполнение deal_owner_name из контактов
CONTACTS_CLEAN = os.path.join('..', 'data', 'cleaned', 'contacts_clean.pkl')
if os.path.exists(CONTACTS_CLEAN):
    contacts = pd.read_pickle(CONTACTS_CLEAN)
    contacts['id'] = contacts['id'].astype('Int64')
    contact_owner_map = contacts.drop_duplicates('id').set_index('id')['contact_owner_name']
    
    mask_isna = df['deal_owner_name'].isna()
    df.loc[mask_isna, 'deal_owner_name'] = df.loc[mask_isna, 'contact_id'].map(contact_owner_map)

# 4. Восполнение course_duration по моде продукта
if 'product' in df.columns and 'course_duration' in df.columns:
    prod_duration_map = df.groupby('product', observed=True)['course_duration'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    df.loc[df['course_duration'].isna(), 'course_duration'] = df.loc[df['course_duration'].isna(), 'product'].map(prod_duration_map)

# Считаем итоги
missing_after = df[COLS_CHECK].isnull().sum()
filled = missing_before - missing_after

print('\nДинамика восполнения пропусков (с учетом анонимных сделок):')
display(pd.DataFrame({
    'Было NaN': missing_before,
    'Восполнено': filled,
    'Осталось NaN': missing_after
}))



Динамика восполнения пропусков (с учетом анонимных сделок):


,Было NaN,Восполнено,Осталось NaN
source,0,0,0
campaign,5524,1154,4370
city,19080,836,18244
level_of_deutsch,20340,600,19740
deal_owner_name,29,29,0
course_duration,18004,0,18004


In [110]:
print("--- РАСПРЕДЕЛЕНИЕ СТАДИЙ (stage) ---")
display(df['stage'].value_counts().to_frame())

--- РАСПРЕДЕЛЕНИЕ СТАДИЙ (stage) ---


,count
stage,
Lost,15742
Call Delayed,2248
Registered on Webinar,2072
Payment Done,858
Waiting For Payment,325
Qualificated,128
Registered on Offline Day,100
Need to Call - Sales,33
Need To Call,31


In [13]:
# Группировка стадий воронки
STAGE_GROUPS = {
    'Lost': 'Lost',
    'Payment Done': 'Won/Paid',
    'Free Education': 'Won/Paid',
    'Waiting For Payment': 'Active Sales',
    'Need To Call': 'Active Sales',
    'Need to Call - Sales': 'Active Sales',
    'Call Delayed': 'Active Sales',
    'Test Sent': 'Active Sales',
    'Registered on Webinar': 'Marketing/Lead',
    'Qualificated': 'Marketing/Lead',
    'Registered on Offline Day': 'Marketing/Lead',
    'New Lead': 'Marketing/Lead',
    'Need a consultation': 'Marketing/Lead'
}

df['stage_group'] = df['stage'].map(STAGE_GROUPS).fillna('Other')
df['stage_group'] = df['stage_group'].astype('category')

print("--- РАСПРЕДЕЛЕНИЕ ГРУПП СТАДИЙ ---")
display(df['stage_group'].value_counts().to_frame())


--- РАСПРЕДЕЛЕНИЕ ГРУПП СТАДИЙ ---


,count
stage_group,
Lost,15742
Active Sales,2662
Marketing/Lead,2328
Won/Paid,859


In [112]:
# Анализ пропущенных значений (или 'Unknown') в разрезе стадий воронки
df_not_lost = df[df['stage_group'] != 'Lost'].copy()

# Считаем пропуски (NaN или 'Unknown') для ключевых полей в разрезе каждой стадии
def count_missing(df_group):
    total = len(df_group)
    return pd.Series({
        'Всего сделок': total,
        'Продукт Unknown/NaN': ((df_group['product'] == 'Unknown') | df_group['product'].isna()).sum(),
        'Длительность NaN': df_group['course_duration'].isna().sum(),
        'Closing Date NaN': df_group['closing_date'].isna().sum(),
        'SLA NaN': df_group['sla'].isna().sum()
    })

status_analysis = df_not_lost.groupby('stage', observed=True).apply(count_missing, include_groups=False)

# Фильтруем только те стадии, где в сумме > 0 сделок
status_analysis = status_analysis[status_analysis['Всего сделок'] > 0]

# Добавляем % для наглядности (только для продукта, так как это ключевой разрез)
status_analysis['% Продукт не указан'] = (status_analysis['Продукт Unknown/NaN'] / status_analysis['Всего сделок'] * 100).round(1)

print("Анализ неполных данных по стадиям (кроме Lost):")
display(status_analysis.sort_values('Всего сделок', ascending=False))

Анализ неполных данных по стадиям (кроме Lost):


,Всего сделок,Продукт Unknown/NaN,Длительность NaN,Closing Date NaN,SLA NaN,% Продукт не указан
stage,,,,,,
Call Delayed,2248,1861,1861,2020,0,82.8
Registered on Webinar,2072,2069,2069,2069,0,99.9
Payment Done,858,17,18,337,0,2.0
Waiting For Payment,325,1,1,296,0,0.3
Qualificated,128,102,102,126,0,79.7
Registered on Offline Day,100,99,99,100,0,99.0
Need to Call - Sales,33,33,33,31,0,100.0
Need To Call,31,31,31,31,0,100.0
Test Sent,25,17,17,23,0,68.0


In [14]:
# Поля, где пропуск = отсутствие информации → заполняем 'Unknown'
FILL_UNKNOWN = ['quality', 'lost_reason', 'campaign', 'content', 'term',
                'payment_type', 'product', 'education_type', 'city', 'level_of_deutsch', 'deal_owner_name']

# Обработка категориальных колонок (предотвращаем TypeError: Categorical)
for col in FILL_UNKNOWN:
    if col in df.columns:
        n_miss = df[col].isna().sum()
        if n_miss > 0:
            # Если колонка категориальная, добавляем категорию перед fillna
            if isinstance(df[col].dtype, pd.CategoricalDtype):
                if 'Unknown' not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories('Unknown')
            else:
                # Если не категориальная, приводим к строке (object), чтобы избежать проблем
                df[col] = df[col].astype(object)
            
            df[col] = df[col].fillna('Unknown')
            print(f'{col}: заполнено {n_miss} пропусков → "Unknown"')

print("\nФинальный статус пропусков в ключевых колонках:")
remaining = df.isnull().sum()
display(
    pd.DataFrame({'Пропуски': remaining, '%': (remaining / len(df) * 100).round(2)})
    .query('Пропуски > 0')
    .sort_values('Пропуски', ascending=False)
)

quality: заполнено 2252 пропусков → "Unknown"
lost_reason: заполнено 5468 пропусков → "Unknown"
campaign: заполнено 4370 пропусков → "Unknown"
content: заполнено 7444 пропусков → "Unknown"
term: заполнено 9137 пропусков → "Unknown"
payment_type: заполнено 21095 пропусков → "Unknown"
product: заполнено 17999 пропусков → "Unknown"
education_type: заполнено 18292 пропусков → "Unknown"
city: заполнено 18244 пропусков → "Unknown"
level_of_deutsch: заполнено 19740 пропусков → "Unknown"

Финальный статус пропусков в ключевых колонках:


,Пропуски,%
months_of_study,20751,96.11
course_duration,18004,83.39
closing_date,6947,32.18


## 9. Категориальные поля

In [15]:
CAT_COLS = [
    'deal_owner_name', 'quality', 'stage', 'lost_reason', 'page',
    'campaign', 'content', 'term', 'source', 'payment_type',
    'product', 'education_type', 'city', 'level_of_deutsch'
]

for col in CAT_COLS:
    if df[col].dtype == object or df[col].dtype.name == 'string':
        df[col] = df[col].str.strip()
    df[col] = df[col].astype('category')
    print(f'{col}: {df[col].nunique()} уникальных')


deal_owner_name: 27 уникальных
quality: 7 уникальных
stage: 13 уникальных
lost_reason: 22 уникальных
page: 34 уникальных
campaign: 155 уникальных
content: 188 уникальных
term: 221 уникальных
source: 13 уникальных
payment_type: 4 уникальных
product: 6 уникальных
education_type: 3 уникальных
city: 877 уникальных
level_of_deutsch: 9 уникальных


In [16]:
# Определение типа контакта: Лид vs Покупатель (Buyer)
# Покупатель — тот, у кого есть хотя бы одна сделка с initial_amount_paid > 0.

# 1. Находим уникальные ID контактов, совершивших оплаты
buyers_ids = df[df['initial_amount_paid'] > 0]['contact_id'].unique()

# 2. Присваиваем статус (Buyer/Lead)
df['contact_type'] = df['contact_id'].isin(buyers_ids).map({True: 'Buyer', False: 'Lead'})
df['contact_type'] = df['contact_type'].astype('category')

print("--- РАСПРЕДЕЛЕНИЕ ТИПОВ КОНТАКТОВ В СДЕЛКАХ ---")
display(df.drop_duplicates('contact_id')['contact_type'].value_counts().to_frame())


--- РАСПРЕДЕЛЕНИЕ ТИПОВ КОНТАКТОВ В СДЕЛКАХ ---


,count
contact_type,
Lead,14877
Buyer,3213


## 11. Итоговый осмотр

In [116]:
h.descr_df(df, include='all', show_sample_rows=True)

,Название признака,Тип данных,Количество значений,Пропуски (NaN),Уникальных значений,Пример строка 1,Пример строка 2,Пример строка 3,Минимум,Среднее,Медиана,Максимум
0,id,Int64,21591,0,8612,5805028000001588224,5805028000004028416,5805028000005183488,5805028000000921600.0,5805028000030050304.0,5805028000030103552.0,5805028000056892416.0
1,deal_owner_name,category,21591,0,28,John Doe,Charlie Davis,Jane Smith,<NA>,<NA>,<NA>,<NA>
2,closing_date,datetime64[us],14644,6947,359,NaT,2023-09-20 00:00:00,2023-08-23 00:00:00,<NA>,<NA>,<NA>,<NA>
3,quality,category,21591,0,7,E - Non Qualified,C - Low,E - Non Qualified,<NA>,<NA>,<NA>,<NA>
4,stage,category,21591,0,13,Lost,Lost,Lost,<NA>,<NA>,<NA>,<NA>
5,lost_reason,category,21591,0,22,Duplicate,Changed Decision,Doesn't Answer,<NA>,<NA>,<NA>,<NA>
6,page,category,21591,0,34,/,/,/,<NA>,<NA>,<NA>,<NA>
7,campaign,category,21591,0,155,nina,nina,nina,<NA>,<NA>,<NA>,<NA>
8,sla,Int32,21591,0,11510,6982,3746,4,3.0,21903.566162,14344.0,86319.0
9,content,category,21591,0,188,Unknown,Unknown,Unknown,<NA>,<NA>,<NA>,<NA>


In [117]:
print(f'Итоговая форма: {df.shape}')
print()
print('Типы данных:')
print(df.dtypes)
print()
df.head()

Итоговая форма: (21591, 25)

Типы данных:
id                              Int64
deal_owner_name              category
closing_date           datetime64[us]
quality                      category
stage                        category
lost_reason                  category
page                         category
campaign                     category
sla                             Int32
content                      category
term                         category
source                       category
payment_type                 category
product                      category
education_type               category
created_time           datetime64[us]
course_duration               float64
months_of_study               float64
initial_amount_paid           float64
offer_total_amount            float64
contact_id                      Int64
city                         category
level_of_deutsch             category
stage_group                  category
contact_type                 category
dtype: o

,id,deal_owner_name,closing_date,quality,stage,lost_reason,page,campaign,sla,content,...,created_time,course_duration,months_of_study,initial_amount_paid,offer_total_amount,contact_id,city,level_of_deutsch,stage_group,contact_type
21478,5805028000001588224,John Doe,NaT,E - Non Qualified,Lost,Duplicate,/,nina,6982,Unknown,...,2023-07-13 10:01:00,NaN,NaN,0.0,0.0,-1,-,A2,Lost,Buyer
20742,5805028000004028416,Charlie Davis,2023-09-20,C - Low,Lost,Changed Decision,/,nina,3746,Unknown,...,2023-08-07 16:52:00,11.0,NaN,0.0,0.0,-1,-,A2,Lost,Buyer
20270,5805028000005183488,Jane Smith,2023-08-23,E - Non Qualified,Lost,Doesn't Answer,/,nina,4,Unknown,...,2023-08-20 19:59:00,NaN,NaN,0.0,0.0,-1,-,A2,Lost,Buyer
20068,5805028000005695488,Jane Smith,2023-09-10,E - Non Qualified,Lost,Stopped Answering,/,nina,6090,Unknown,...,2023-08-24 10:37:00,NaN,NaN,0.0,0.0,-1,-,A2,Lost,Buyer
20062,5805028000005695488,Jane Smith,2023-10-14,C - Low,Lost,Stopped Answering,/,nina,12763,Unknown,...,2023-08-24 12:51:00,NaN,NaN,0.0,0.0,-1,-,A2,Lost,Buyer


## 12. Сохранение

In [17]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
df.to_pickle(OUT_PATH)

summary_data = {
    'Метрика': [
        'Строк исходно',
        'Строк после очистки',
        'Удалено дубликатов',
        'Уникальных сделок (id)',
        'Диапазон created_time',
        'stage (уникальных)',
        'Медиана SLA, мин',
        'Пропуски после заполнения'
    ],
    'Значение': [
        n_before,
        len(df),
        n_before - len(df),
        df['id'].nunique(),
        f'{df["created_time"].min().date()} → {df["created_time"].max().date()}',
        df['stage'].nunique(),
        f'{df["sla"].median() / 60:.1f}',
        df.isnull().sum().sum()
    ]
}

print(f'Сохранено: {OUT_PATH}')
display(pd.DataFrame(summary_data))

Сохранено: ..\data\cleaned\deals_clean.pkl


,Метрика,Значение
0,Строк исходно,21595
1,Строк после очистки,21591
2,Удалено дубликатов,4
3,Уникальных сделок (id),8612
4,Диапазон created_time,2023-07-03 → 2024-06-21
5,stage (уникальных),13
6,"Медиана SLA, мин",239.1
7,Пропуски после заполнения,45702


In [18]:
# Формируем и экспортируем данные по покупателям для блокнота Контактов (01)
# Передаем дату первой оплаты, чтобы верно определить момент превращения Лида в Клиента.

BUYERS_INFO_PATH = os.path.join('..', 'data', 'cleaned', 'buyers_info.pkl')

buyers_data = (
    df[df['contact_type'] == 'Buyer']
    .groupby('contact_id')
    .agg(first_payment_date=('created_time', 'min'))
    .reset_index()
)

# Сохраняем для использования в 01_cleaning_contacts.ipynb
buyers_data.to_pickle(BUYERS_INFO_PATH)

print(f"Информация о {len(buyers_data)} покупателях экспортирована в {BUYERS_INFO_PATH}")


Информация о 3213 покупателях экспортирована в ..\data\cleaned\buyers_info.pkl


## Описание датасета

**Источник:** `Deals (Done).xlsx` — выгрузка сделок из Zoho CRM  
**Назначение:** основной датасет для анализа выручки, стадий продаж, воронки и LTV

### Ключевые аналитические показатели
| Столбец | Тип | Описание |
|---|---|---|
| `id` | `int64` | Уникальный ID сделки в CRM (19-значный) |
| `contact_id` | `int64` | ID связанного контакта (связь с `contacts.id`). Для анонимных сделок = `-1` |
| `deal_owner_name` | `category` | Менеджер, ответственный за закрытие сделки |
| `stage` | `category` | Исходная стадия воронки в CRM |
| `stage_group` | `category` | **Группировка стадий:** Paid, Lost, Active Sales, Marketing |
| `initial_amount_paid` | `float64` | **Фактическая выручка:** оплаченная сумма |
| `offer_total_amount` | `float64` | Полная контрактная стоимость продукта |
| `created_time` | `datetime` | Дата и время создания сделки |
| `closing_date` | `datetime` | Дата завершения сделки (факт профита или отказа) |
| `sla` | `int32` | **Время ответа:** разница в секундах между регистрацией и действием |
| `contact_type` | `category` | **Тип клиента:** Buyer (с оплатами) / Lead (без оплат) |

### Обработка пропусков и специфика колонок
В процессе очистки была проведена работа по восполнению данных (**Backfill**) и нормализации.

- **Нормализация уровня языка (`level_of_deutsch`):**
    - Исходные данные содержали 215 вариантов написания (кириллица, латиница, текст).
    - Применены регулярные выражения и маппинг кириллицы (`а2` -> `A2`, `б1` -> `B1`).
    - Значения приведены к международному стандарту CEFR (A0–C2). Неразборчивые записи помечены как `Unclear`.

- **Восполнение (Backfill):** 
    - `source`, `campaign`, `city`, `level_of_deutsch`, `deal_owner_name`: данные "протянуты" между сделками одного и того же клиента (по `contact_id`). Если у клиента в одной сделке был указан уровень языка или город, а в другой — нет, мы восстановили эти данные.
    - `course_duration`: частично восполнено на основе выбранного продукта (`product`).

- **Оставлено "как есть" (NaN):**
    - `course_duration` (~83% пропусков) и `months_of_study` (~96% пропусков).
    - **Причина:** Данные поля заполняются в CRM преимущественно для успешных сделок (`Won/Paid`). Попытка заполнить их для лидов (через среднее или моду) приведет к серьезному искажению аналитики по продуктовой линейке и LTV. 
    - **closing_date** (~32% пропусков): Отсутствие даты означает, что сделка всё еще находится в работе (не закрыта ни в плюс, ни в минус).

- **Заполнено "Unknown":**
    - Категориальные поля (`quality`, `product`, `payment_type` и др.), где отсутствие информации является результатом отсутствия ввода данных в CRM.

**Ключевые связи:**
- `contact_id` → [01_cleaning_contacts.ipynb](01_cleaning_contacts.ipynb) (`id`)
- `closing_date` → основной временной ряд для финансовой аналитики
- `stage_group` → фильтр для конверсии и ROMI